# Whisper Bangla LoRA — Colab Debug Run

Smoke-tests all 7 steps on a **15 GB T4** with 100 samples / 5 training steps.
Every box in the checklist must pass before the DGX full run.

**This run does not train large-v3.** A T4 is Turing: no bfloat16, and 15 GB is
too tight for a 1.55B model plus activations. The debug profile trains
`openai/whisper-medium` (769M) in fp16 instead — same code path, same collator,
same adapter shape, ~2x smaller and fp16-capable. Only the profile changes on the DGX.

Runtime → Change runtime type → **T4 GPU**.


## 0 · Setup

In [ ]:
!nvidia-smi
!git clone https://github.com/<you>/whisper-finetuning.git repo || true
%cd repo
# Prefer the compiled lock; fall back to direct pins if it hasn't been built yet.
!test -f requirements/train-linux-cu124.txt   && pip install -q -r requirements/train-linux-cu124.txt   || pip install -q -r requirements/train.in


In [ ]:
import os, sys
from getpass import getpass
os.environ["HF_TOKEN"] = getpass("HF token: ")   # needed for gated Common Voice
sys.path.insert(0, "scripts")


## 1 · Explore datasets
☐ all 4 sources load without error

In [ ]:
!python scripts/01_explore_datasets.py

## 2 · Prepare data (100 samples each)
☐ `process_*` returns the unified schema

In [ ]:
for src in ["medibeng", "indicvoices", "common_voice", "fleurs"]:
    !python scripts/02_prepare_data.py --source {src} --limit 100


## 3 · Augment MediBeng
☐ augmented audio sounds reasonable — listen below

In [ ]:
!python scripts/03_augment.py --source medibeng --n-augments 4

In [ ]:
from datasets import load_from_disk
from IPython.display import Audio, display
aug = load_from_disk("data/augmented/medibeng")
for i in range(2):
    print(aug[i]["source"], "|", aug[i]["sentence"][:80])
    display(Audio(aug[i]["audio"]["array"], rate=16000))


## 4 · Build final dataset
☐ train/val/test splits with sane sizes

In [ ]:
!python scripts/04_build_dataset.py

## 5 · Train (whisper-medium, 5 steps, r=64)
☐ loss appears, no NaN/inf, no OOM  ☐ checkpoint saves

Effective batch is **2 x 8 accumulation = 16**; `max_steps: 5` counts optimizer
steps, so this is 80 forward passes. If the T4 OOMs, drop to
`--model openai/whisper-small` rather than cutting the accumulation — the whole
point is to exercise the accumulation path the DGX run also uses.


In [ ]:
!python scripts/05_train.py
# if the T4 OOMs:  !python scripts/05_train.py --model openai/whisper-small
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv


## 6 · Eval on 10 test samples
☐ WER is computable

In [ ]:
!python scripts/06_eval.py --adapter outputs/whisper-bangla-lora-debug --limit 10

## 7 · Export to faster-whisper
☐ export runs and faster-whisper loads the result

Merging medium needs ~6 GB host RAM — fine on Colab. The large-v3 merge needs
~12 GB and belongs on the DGX.


In [ ]:
!python scripts/07_export.py --adapter outputs/whisper-bangla-lora-debug

In [ ]:
from faster_whisper import WhisperModel
m = WhisperModel("outputs/faster-whisper-bangla", device="cuda", compute_type="float16")
print("loaded OK")


All boxes ticked → move to `full_train_dgx.ipynb`, which runs the identical
scripts with `--full` (large-v3, bf16, r=128, effective batch 64).

What this debug run does **not** prove: large-v3 memory headroom, bf16 numerics,
or absolute WER. Feature dimensionality also differs (80 mel bins here vs 128 on
large-v3), so rebuild features on the DGX rather than copying a cache.
